In [ ]:
# ==========================================
# WEEK 1: THE MASTER MEDICAL AI PIPELINE
# ==========================================
import pydicom
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pydicom.data import get_testdata_files
from skimage import measure, morphology
from scipy import ndimage

# 1. LOAD & CALIBRATE (Days 1-2)
filename = get_testdata_files("CT_small.dcm")[0]
ds = pydicom.dcmread(filename)
hu_image = ds.pixel_array * ds.RescaleSlope + ds.RescaleIntercept

# 2. SEGMENT & CLEAN (Days 3-4)
# Thresholding for soft tissue
soft_tissue_mask = (hu_image > -100) & (hu_image < 60)

# Connected Components to find the patient and remove noise
labels = measure.label(soft_tissue_mask)
props = measure.regionprops(labels)
largest_component = props[np.argmax([p.area for p in props])]
largest_only_mask = (labels == largest_component.label)

# 3. 3D REFINEMENT (Day 5)
# Create a 3D volume by stacking (simulating a full scan)
volume_stack = np.stack([largest_only_mask] * 10, axis=0)

# Fill internal holes and smooth the edges with a 3D ball
filled_volume = ndimage.binary_fill_holes(volume_stack)
refined_3d = morphology.binary_closing(filled_volume, morphology.ball(3))

# 4. EXPORT TO NIfTI (Day 5 Wrap-up)
# Convert to float for AI compatibility and save with an Affine matrix
nifti_data = refined_3d.astype(np.float32)
nifti_obj = nib.Nifti1Image(nifti_data, affine=np.eye(4))
nib.save(nifti_obj, "week1_final_output.nii.gz")

# 5. VISUALIZE THE FINAL PRODUCT
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(hu_image, cmap='gray')
plt.title("Raw Input (HU)")
plt.subplot(1, 2, 2)
plt.imshow(refined_3d[5], cmap='magma') # Showing the 5th slice of the 3D volume
plt.title("Final 3D Refined Mask")
plt.show()

print("✅ Week 1 Pipeline Complete! 'week1_final_output.nii.gz' is ready for AI Training.")